# Mind2Web Baselines: Multimodal + CoT & AXTree Only
### Model: Qwen3-VL-8B-Instruct

This notebook runs two baselines:
1. **Multimodal + CoT** — screenshot + DOM text, with chain-of-thought reasoning
2. **AXTree only** — accessibility tree text, no image, no CoT

---
**First time setup** — run this in your EC2 terminal before starting:
```bash
pip install transformers accelerate bitsandbytes qwen-vl-utils datasets pillow tqdm
```

## Cell 1 — Imports & Config

In [1]:
import os, json, gc, re
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from datasets import load_dataset

# Reduce CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HOME"] = "/home/ubuntu/hf_cache"

CONFIG = {
    "model_id":        "Qwen/Qwen3-VL-8B-Instruct",
    "results_dir":     "/home/ubuntu/results",
    "device":          "cuda" if torch.cuda.is_available() else "cpu",
    "dtype":           torch.bfloat16,
    "max_new_tokens":  256,    # for AXTree baseline
    "cot_max_tokens":  128,    # for CoT baseline (needs more room to reason)
    "max_samples":     None,    # set to None for full dataset
    "dataset_split":   "test_website",
    "MAX_DOM_CHARS":   2000,   # truncate DOM to avoid exceeding context window
    "MAX_AX_CHARS":    2000,   # truncate AXTree
}

Path(CONFIG["results_dir"]).mkdir(parents=True, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

/opt/pytorch/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


PyTorch : 2.9.1+cu130
CUDA    : True
GPU     : NVIDIA A10G
VRAM    : 23.7 GB


In [27]:
CONFIG = {
    "model_id":        "Qwen/Qwen3-VL-8B-Instruct",
    "results_dir":     "/home/ubuntu/results",
    "device":          "cuda" if torch.cuda.is_available() else "cpu",
    "dtype":           torch.bfloat16,
    "max_new_tokens":  256,    # for AXTree baseline
    "cot_max_tokens":  512,    # for CoT baseline (needs more room to reason)
    "max_samples":     None,    # set to None for full dataset
    "dataset_split":   "test_website",
    "MAX_DOM_CHARS":   2000,   # truncate DOM to avoid exceeding context window
    "MAX_AX_CHARS":    2000,   # truncate AXTree
}

## Cell 2 — Dataset

In [2]:
import io

def _parse_candidate(c):
    if isinstance(c, str):
        try:
            return json.loads(c)
        except json.JSONDecodeError:
            return {"backend_node_id": c, "tag_name": "", "text": c}
    return c

def _parse_operation(op):
    if isinstance(op, str):
        try:
            return json.loads(op)
        except json.JSONDecodeError:
            return {"op": op, "value": ""}
    return op or {"op": None, "value": ""}


class Mind2WebDataset:
    """
    Streaming loader for osunlp/Multimodal-Mind2Web.
    Splits: train | test_task | test_website | test_domain
    """
    def __init__(self, split="test_website"):
        self.split = split
        self.ds = load_dataset(
            "osunlp/Multimodal-Mind2Web", split=split, streaming=True
        )

    def __iter__(self):
        for row in self.ds:
            yield self._process(row)

    def _process(self, row):
        pos_raw = row.get("pos_candidates") or []
        neg_raw = row.get("neg_candidates") or []
        pos = [_parse_candidate(c) for c in pos_raw]
        neg = [_parse_candidate(c) for c in neg_raw]

        def normalise(c):
            return {
                "element_id": c.get("backend_node_id", ""),
                "tag":        c.get("tag_name", c.get("tag", "")),
                "text":       c.get("text", ""),
            }

        op = _parse_operation(row.get("operation"))

        screenshot = row.get("image")
        if isinstance(screenshot, dict) and "bytes" in screenshot:
            screenshot = Image.open(io.BytesIO(screenshot["bytes"])).convert("RGB")

        return {
            "action_uid":         row.get("action_uid", ""),
            "instruction":        row.get("confirmed_task", ""),
            "dom_text":           row.get("cleaned_html", ""),
            "axtree_text":        row.get("axtree", ""),
            "screenshot":         screenshot,
            "candidate_elements": [normalise(c) for c in pos + neg],
            "label_element_id":   pos[0].get("backend_node_id") if pos else None,
            "label_action":       op.get("op"),
            "label_value":        op.get("value", ""),
        }

    def get_sample(self):
        return next(iter(self))


# Quick sanity check
print("Loading one sample to verify dataset...")
dataset = Mind2WebDataset(split=CONFIG["dataset_split"])
sample = dataset.get_sample()
print(f"  Task       : {sample['instruction']}")
print(f"  Action     : {sample['label_action']} on {sample['label_element_id']}")
print(f"  Candidates : {len(sample['candidate_elements'])}")
print(f"  Screenshot : {type(sample['screenshot'])} {getattr(sample['screenshot'], 'size', '')}")
print(f"  DOM length : {len(sample['dom_text'])} chars")
print(f"  AXTree     : {len(sample['axtree_text'])} chars" if sample['axtree_text'] else "  AXTree     : (empty)")
print("Dataset OK.")

Loading one sample to verify dataset...


Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

  Task       : What are the romantic reggae musics from BCD Studio that can be used in tik tok series in andorra
  Action     : CLICK on 110
  Candidates : 493
  Screenshot : <class 'NoneType'> 
  DOM length : 85102 chars
  AXTree     : (empty)
Dataset OK.


## Cell 3 — AXTree Conversion

If `axtree_text` comes back empty from the dataset, this cell converts the DOM HTML into a lightweight accessibility-tree-style representation.

In [3]:
from html.parser import HTMLParser

INTERACTIVE_TAGS = {"a", "button", "input", "select", "textarea", "label", "option"}

class DOMToAXTree(HTMLParser):
    """
    Converts raw HTML into a readable AXTree-style text representation.
    Only keeps interactive elements + headings to stay within token limits.
    """
    def __init__(self):
        super().__init__()
        self.lines = []
        self.depth = 0
        self.current_tag = None
        self.current_text = ""
        self.current_attrs = {}

    def handle_starttag(self, tag, attrs):
        self.depth += 1
        self.current_tag = tag
        self.current_attrs = dict(attrs)
        self.current_text = ""  # reset text buffer on new tag

    def handle_endtag(self, tag):
        if tag in INTERACTIVE_TAGS or tag in {"h1","h2","h3"}:
            attrs = self.current_attrs
            # Try more attribute sources specific to Mind2Web
            name = (attrs.get("aria-label") or
                    attrs.get("placeholder") or
                    attrs.get("aria-description") or
                    attrs.get("value") or
                    attrs.get("alt") or
                    attrs.get("title") or
                    self.current_text.strip()[:80] or
                    attrs.get("class", "")[:40] or
                    "(unlabelled)")

            # Mind2Web uses backend_node_id as the element identifier
            node_id = (attrs.get("backend_node_id") or 
                       attrs.get("id") or "")

            indent = "  " * min(self.depth, 6)
            self.lines.append(f"{indent}{tag}[name='{name}', id={node_id}]")
        self.depth = max(0, self.depth - 1)

    def handle_data(self, data):
        self.current_text += data

    def get_axtree(self):
        return "\n".join(self.lines)


def dom_to_axtree(html: str, max_chars: int = 6000) -> str:
    """Convert HTML string to AXTree text, truncated to max_chars."""
    if not html:
        return ""
    parser = DOMToAXTree()
    try:
        parser.feed(html[:50000])  # limit input to parser too
        result = parser.get_axtree()
    except Exception:
        result = ""
    return result[:max_chars]


# Test it on our sample
if sample["axtree_text"]:
    print("AXTree already present in dataset — no conversion needed.")
    print(sample["axtree_text"][:300])
else:
    print("AXTree not in dataset — converting from DOM...")
    ax = dom_to_axtree(sample["dom_text"])
    print(f"Generated AXTree ({len(ax)} chars):")
    print(ax[:300])

AXTree not in dataset — converting from DOM...
Generated AXTree (6000 chars):
            a[name='Tiktok for business', id=106]
            a[name='navArrow', id=205]
            a[name='navArrow', id=210]
            a[name='navArrow', id=216]
            a[name='navArrow', id=221]
            a[name='navArrow', id=226]
            a[name='navArrow', id=236]
            a[na


## Cell 4 — Prompt Templates

In [17]:
def format_candidates(candidate_elements):
    lines = []
    for i, el in enumerate(candidate_elements):
        lines.append(f"[{i}] id={el['element_id']} tag={el['tag']} text=\"{el['text']}\"")
    return "\n".join(lines)


SYSTEM_COT = """You are a web agent. Given a task and webpage information, predict the next action.
Keep your reasoning to 2-3 sentences maximum.
You MUST end your response with a JSON on the last line, no matter what:
{"element_id": "<id>", "action": "<click|type|select>", "value": "<optional>"}
If you are unsure, make your best guess but always output the JSON."""


SYSTEM_AXTREE = """You are a web agent. Given a task and an accessibility tree of the webpage, predict the next action.
Respond ONLY in JSON format: {"element_id": "<id>", "action": "<click|type|select>", "value": "<optional>"}
Do not include any explanation outside the JSON."""


def make_prompt_cot(sample):
    """Baseline: Multimodal (image + DOM) with Chain-of-Thought."""
    dom  = sample["dom_text"][:CONFIG["MAX_DOM_CHARS"]]
    cands = format_candidates(sample["candidate_elements"])
    text = (
        f"{SYSTEM_COT}\n\n"
        f"Task: {sample['instruction']}\n\n"
        f"DOM:\n{dom}\n\n"
        f"Candidate elements:\n{cands}\n\n"
        "The screenshot of the current webpage is provided.\n"
        "Reasoning and next action:"
    )
    return text, sample["screenshot"]   # returns (prompt, image)


def make_prompt_axtree(sample):
    """Baseline: AXTree text only (no image)."""
    # Use dataset axtree if available, otherwise convert from DOM
    ax = sample["axtree_text"] or dom_to_axtree(sample["dom_text"])
    ax = ax[:CONFIG["MAX_AX_CHARS"]]
    cands = format_candidates(sample["candidate_elements"])
    text = (
        f"{SYSTEM_AXTREE}\n\n"
        f"Task: {sample['instruction']}\n\n"
        f"Accessibility Tree:\n{ax}\n\n"
        f"Candidate elements:\n{cands}\n\n"
        "Next action:"
    )
    return text, None   # no image for this baseline


# Quick prompt preview
p_cot,  img_cot  = make_prompt_cot(sample)
p_ax,   img_ax   = make_prompt_axtree(sample)
print("CoT prompt (first 400 chars):")
print(p_cot[:400])
print("\nAXTree prompt (first 400 chars):")
print(p_ax[:2000])

CoT prompt (first 400 chars):
You are a web agent. Given a task and webpage information, predict the next action.
Keep your reasoning to 2-3 sentences maximum.
You MUST end your response with a JSON on the last line, no matter what:
{"element_id": "<id>", "action": "<click|type|select>", "value": "<optional>"}
If you are unsure, make your best guess but always output the JSON.

Task: What are the romantic reggae musics from BC

AXTree prompt (first 400 chars):
You are a web agent. Given a task and an accessibility tree of the webpage, predict the next action.
Respond ONLY in JSON format: {"element_id": "<id>", "action": "<click|type|select>", "value": "<optional>"}
Do not include any explanation outside the JSON.

Task: What are the romantic reggae musics from BCD Studio that can be used in tik tok series in andorra

Accessibility Tree:
            a[name='Tiktok for business', id=106]
            a[name='navArrow', id=205]
            a[name='navArrow', id=210]
            a[name='nav

## Cell 5 — Load Model

In [5]:
from qwen_vl_utils import process_vision_info


from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch

model_id = "Qwen/Qwen3-VL-8B-Instruct"

# 4-bit quantization config (best balance of speed + quality)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# FORCE model onto the single GPU
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="cuda:0",   # IMPORTANT: do NOT use "auto"
)

model.eval()
processor = AutoProcessor.from_pretrained(model_id)

print("Model ready (4-bit on A10G).")
print("Model device:", next(model.parameters()).device)

if torch.cuda.is_available():
    print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"VRAM reserved : {torch.cuda.memory_reserved()/1e9:.2f} GB")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Model ready (4-bit on A10G).
Model device: cuda:0
VRAM allocated: 6.38 GB
VRAM reserved : 16.56 GB


In [6]:
import transformers
print(transformers.__version__)
print(transformers.__file__)

4.40.2
/opt/pytorch/lib/python3.12/site-packages/transformers/__init__.py


## Cell 6 — Inference Function

In [6]:
@torch.inference_mode()
def predict(text_prompt, image=None, max_new_tokens=256):
    """
    Run one forward pass.
    image: PIL.Image or None (text-only if None)
    """
    content = []
    if image is not None:
        content.append({"type": "image", "image": image})
    content.append({"type": "text", "text": text_prompt})

    messages = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        return_tensors="pt",
    ).to(CONFIG["device"])

    generated = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        use_cache=True,
    )
    out = processor.decode(
        generated[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )
    return out.strip()


# Smoke test
print("Running smoke test...")
test_out = predict("Reply with: OK", image=None, max_new_tokens=10)
print(f"Model response: {test_out}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Running smoke test...
Model response: OK


## Cell 7 — Parsing & Evaluation

In [7]:
def parse_prediction(raw_output):
    """
    Extract JSON from model output.
    For CoT, the JSON will be at the end after the reasoning.
    """
    matches = re.findall(r'\{[^{}]*\}', raw_output, re.DOTALL)
    for match in reversed(matches):  # last JSON block = final answer
        try:
            parsed = json.loads(match)
            if "element_id" in parsed and "action" in parsed:
                parsed.setdefault("value", "")
                return parsed
        except json.JSONDecodeError:
            continue
    return None


def evaluate(pred, sample):
    if pred is None:
        return {"element_acc": 0, "action_acc": 0, "exact_match": 0, "parse_fail": 1}
    element_correct = str(pred["element_id"]) == str(sample["label_element_id"])
    action_correct  = pred["action"].lower() == str(sample["label_action"]).lower()
    return {
        "element_acc": int(element_correct),
        "action_acc":  int(action_correct),
        "exact_match": int(element_correct and action_correct),
        "parse_fail":  0,
    }


def aggregate(results):
    keys = ["element_acc", "action_acc", "exact_match", "parse_fail"]
    return {k: float(np.mean([r[k] for r in results])) for k in keys} | {"n": len(results)}


print("Evaluation functions ready.")

Evaluation functions ready.


## Cell 8 — Main Loop

In [28]:
# def run_baseline(baseline_name, prompt_fn, max_new_tokens, split=CONFIG["dataset_split"]):
#     """
#     baseline_name : used for output filenames
#     prompt_fn     : make_prompt_cot or make_prompt_axtree
#     max_new_tokens: use cot_max_tokens for CoT, max_new_tokens for AXTree
#     """
#     max_samples = CONFIG["max_samples"]
#     results_dir = Path(CONFIG["results_dir"])

#     all_results = []
#     raw_outputs = []

#     ds = Mind2WebDataset(split=split)  # fresh iterator
#     pbar = tqdm(desc=baseline_name, total=max_samples)

#     for i, sample in enumerate(ds):
#         if max_samples and i >= max_samples:
#             break

#         prompt, image = prompt_fn(sample)

#         try:
#             raw = predict(prompt, image=image, max_new_tokens=max_new_tokens)
#         except Exception as e:
#             print(f"  [{i}] Error: {e}")
#             raw = ""
        

#         pred    = parse_prediction(raw)
#         metrics = evaluate(pred, sample)
#         all_results.append(metrics)

#         raw_outputs.append({
#             "idx":        i,
#             "task_id":    sample.get("action_uid", str(i)),
#             "instruction": sample["instruction"],
#             "raw":        raw,
#             "parsed":     pred,
#             "gt_element": sample["label_element_id"],
#             "gt_action":  sample["label_action"],
#             **metrics,
#         })

#         # Checkpoint every 50 samples
#         if (i + 1) % 50 == 0:
#             cp = results_dir / f"{baseline_name}_checkpoint_{i+1}.jsonl"
#             with open(cp, "w") as f:
#                 for row in raw_outputs:
#                     f.write(json.dumps(row) + "\n")
#             tqdm.write(f"  Checkpoint saved at {i+1} samples")

#         pbar.update(1)

#     pbar.close()

#     # Save final outputs
#     with open(results_dir / f"{baseline_name}_raw.jsonl", "w") as f:
#         for row in raw_outputs:
#             f.write(json.dumps(row) + "\n")

#     summary = aggregate(all_results)
#     with open(results_dir / f"{baseline_name}_summary.json", "w") as f:
#         json.dump({"baseline": baseline_name, **summary}, f, indent=2)

#     print(f"\n{'='*50}")
#     print(f"Baseline    : {baseline_name}")
#     print(f"N samples   : {summary['n']}")
#     print(f"Element Acc : {summary['element_acc']:.3f}  ({int(summary['element_acc']*summary['n'])}/{summary['n']})")
#     print(f"Action Acc  : {summary['action_acc']:.3f}  ({int(summary['action_acc']*summary['n'])}/{summary['n']})")
#     print(f"Exact Match : {summary['exact_match']:.3f}  ({int(summary['exact_match']*summary['n'])}/{summary['n']})")
#     print(f"Parse Fail  : {summary['parse_fail']:.3f}  ({int(summary['parse_fail']*summary['n'])}/{summary['n']})")
#     print(f"{'='*50}\n")

#     return summary


# print("run_baseline() ready.")

run_baseline() ready.


In [31]:
from collections import defaultdict
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm


def compute_full_metrics(raw_outputs):
    """
    Computes the full set of metrics matching the paper format.
    Requires raw_outputs to contain per-sample predictions and ground truth.
    """
    num_lines   = len(raw_outputs)
    num_parsed  = sum(1 for r in raw_outputs if r["parsed"] is not None)
    parse_fail  = 1.0 - (num_parsed / num_lines) if num_lines > 0 else 0.0

    # ── Element Accuracy ────────────────────────────────────────────────────
    element_correct = [
        int(str(r["parsed"]["element_id"]) == str(r["gt_element"]))
        if r["parsed"] else 0
        for r in raw_outputs
    ]
    element_accuracy = float(np.mean(element_correct))

    # ── Action Accuracy ──────────────────────────────────────────────────────
    action_correct = [
        int(r["parsed"]["action"].upper() == str(r["gt_action"]).upper())
        if r["parsed"] else 0
        for r in raw_outputs
    ]
    action_accuracy = float(np.mean(action_correct))

    # ── Exact Match ──────────────────────────────────────────────────────────
    exact_match = float(np.mean([
        int(e and a) for e, a in zip(element_correct, action_correct)
    ]))

    # ── Top-3 Element Accuracy ───────────────────────────────────────────────
    # Uses candidate ranking: checks if gt element appears in top-3 candidates
    # by checking if the model's predicted element_id matches any of the
    # first 3 candidates in the candidate list
    top3_correct = []
    for r in raw_outputs:
        if r["parsed"] is None:
            top3_correct.append(0)
            continue
        # Get top 3 candidate ids from the stored candidates
        top3_ids = [str(c) for c in r.get("top3_candidate_ids", [])]
        gt = str(r["gt_element"])
        top3_correct.append(int(gt in top3_ids) if top3_ids else element_correct[raw_outputs.index(r)])
    top3_element_accuracy = float(np.mean(top3_correct))

    # ── MRR (Mean Reciprocal Rank) ───────────────────────────────────────────
    mrr_scores = []
    for r in raw_outputs:
        if r["parsed"] is None:
            mrr_scores.append(0.0)
            continue
        candidates = r.get("all_candidate_ids", [])
        gt = str(r["gt_element"])
        pred_id = str(r["parsed"]["element_id"])
        if pred_id == gt:
            # rank 1 if exact match
            mrr_scores.append(1.0)
        elif gt in [str(c) for c in candidates]:
            # find rank of gt in candidates ordered by whether pred matches
            try:
                rank = [str(c) for c in candidates].index(gt) + 1
                mrr_scores.append(1.0 / rank)
            except ValueError:
                mrr_scores.append(0.0)
        else:
            mrr_scores.append(0.0)
    mrr = float(np.mean(mrr_scores))

    # ── Per-Action Metrics ───────────────────────────────────────────────────
    action_types = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0, "support": 0})
    for r in raw_outputs:
        gt_action = str(r["gt_action"]).upper() if r["gt_action"] else "UNKNOWN"
        action_types[gt_action]["support"] += 1
        if r["parsed"]:
            pred_action = r["parsed"]["action"].upper()
            if pred_action == gt_action:
                action_types[gt_action]["tp"] += 1
            else:
                action_types[gt_action]["fn"] += 1
                action_types[pred_action]["fp"] += 1
        else:
            action_types[gt_action]["fn"] += 1

    per_action = {}
    for action, counts in action_types.items():
        tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0.0)
        per_action[action] = {
            "precision": precision,
            "recall":    recall,
            "f1":        f1,
            "support":   counts["support"],
        }

    # ── Task Success Rate ────────────────────────────────────────────────────
    # Groups by instruction (task) — task succeeds if ALL steps in that task
    # have exact match = 1
    task_results = defaultdict(list)
    for r in raw_outputs:
        task_key = r.get("instruction", r.get("task_id", str(r["idx"])))
        em = int(
            r["parsed"] is not None and
            str(r["parsed"]["element_id"]) == str(r["gt_element"]) and
            r["parsed"]["action"].upper() == str(r["gt_action"]).upper()
        )
        task_results[task_key].append(em)

    task_successes = [int(all(steps)) for steps in task_results.values()]
    task_success_rate = float(np.mean(task_successes)) if task_successes else 0.0

    return {
        "num_lines":             num_lines,
        "num_parsed":            num_parsed,
        "parse_failure_rate":    parse_fail,
        "element_accuracy":      element_accuracy,
        "action_accuracy":       action_accuracy,
        "exact_match":           exact_match,
        "top_3_element_accuracy": top3_element_accuracy,
        "mrr":                   mrr,
        "per_action":            per_action,
        "task_success_rate":     task_success_rate,
    }


def run_baseline(baseline_name, prompt_fn, max_new_tokens, split=CONFIG["dataset_split"]):
    max_samples = CONFIG["max_samples"]
    results_dir = Path(CONFIG["results_dir"])
    all_results = []
    raw_outputs = []

    ds   = Mind2WebDataset(split=split)
    pbar = tqdm(desc=baseline_name, total=max_samples)

    for i, sample in enumerate(ds):
        if max_samples and i >= max_samples:
            break

        prompt, image = prompt_fn(sample)

        try:
            raw = predict(prompt, image=image, max_new_tokens=max_new_tokens)
        except Exception as e:
            print(f"  [{i}] Error: {e}")
            raw = ""
        finally:
            torch.cuda.empty_cache()

        pred    = parse_prediction(raw)
        metrics = evaluate(pred, sample)
        all_results.append(metrics)

        # Store candidate ids for top-3 / MRR calculation
        all_candidate_ids = [str(c["element_id"]) for c in sample.get("candidate_elements", [])]
        top3_candidate_ids = all_candidate_ids[:3]

        raw_outputs.append({
            "idx":               i,
            "task_id":           sample.get("action_uid", str(i)),
            "instruction":       sample["instruction"],
            "raw":               raw,
            "parsed":            pred,
            "gt_element":        sample["label_element_id"],
            "gt_action":         sample["label_action"],
            "all_candidate_ids": all_candidate_ids,
            "top3_candidate_ids": top3_candidate_ids,
            **metrics,
        })

        # Checkpoint every 50 samples — saves to disk so progress survives kernel death
        if (i + 1) % 50 == 0:
            cp = results_dir / f"{baseline_name}_checkpoint_{i+1}.jsonl"
            with open(cp, "w") as f:
                for row in raw_outputs:
                    f.write(json.dumps(row) + "\n")
            tqdm.write(f"  Checkpoint saved at {i+1} samples")

        pbar.update(1)

    pbar.close()

    # ── Save raw outputs ─────────────────────────────────────────────────────
    raw_path = results_dir / f"{baseline_name}_raw.jsonl"
    with open(raw_path, "w") as f:
        for row in raw_outputs:
            f.write(json.dumps(row) + "\n")

    # ── Compute & save full metrics ──────────────────────────────────────────
    full_metrics = compute_full_metrics(raw_outputs)
    full_metrics["baseline"] = baseline_name

    metrics_path = results_dir / f"{baseline_name}_metrics.json"
    with open(metrics_path, "w") as f:
        json.dump(full_metrics, f, indent=2)

    # ── Print summary ────────────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"Baseline          : {baseline_name}")
    print(f"Num samples       : {full_metrics['num_lines']}")
    print(f"Parse failure     : {full_metrics['parse_failure_rate']:.3f}  ({full_metrics['num_lines'] - full_metrics['num_parsed']}/{full_metrics['num_lines']})")
    print(f"Element Acc       : {full_metrics['element_accuracy']:.3f}")
    print(f"Action Acc        : {full_metrics['action_accuracy']:.3f}")
    print(f"Exact Match       : {full_metrics['exact_match']:.3f}")
    print(f"Top-3 Element Acc : {full_metrics['top_3_element_accuracy']:.3f}")
    print(f"MRR               : {full_metrics['mrr']:.3f}")
    print(f"Task Success Rate : {full_metrics['task_success_rate']:.3f}")
    print(f"\nPer-action breakdown:")
    for action, s in sorted(full_metrics["per_action"].items()):
        print(f"  {action:<8} P={s['precision']:.3f}  R={s['recall']:.3f}  F1={s['f1']:.3f}  support={s['support']}")
    print(f"{'='*55}\n")
    print(f"Metrics saved to: {metrics_path}")

    return full_metrics

## Cell 9 — Run Baseline: Multimodal + CoT

In [32]:
summary_cot = run_baseline(
    baseline_name  = "multimodal_cot_qwen3",
    prompt_fn      = make_prompt_cot,
    max_new_tokens = CONFIG["cot_max_tokens"],
)

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]




multimodal_cot_qwen3: 0it [00:00, ?it/s]


multimodal_cot_qwen3: 1it [00:10, 10.61s/it]


multimodal_cot_qwen3: 2it [00:19,  9.51s/it]


multimodal_cot_qwen3: 3it [00:28,  9.24s/it]


multimodal_cot_qwen3: 4it [00:37,  9.19s/it]


multimodal_cot_qwen3: 5it [00:45,  8.68s/it]


multimodal_cot_qwen3: 6it [00:53,  8.68s/it]


multimodal_cot_qwen3: 7it [01:02,  8.70s/it]


multimodal_cot_qwen3: 8it [01:12,  8.96s/it]


multimodal_cot_qwen3: 9it [01:16,  7.51s/it]


multimodal_cot_qwen3: 10it [01:23,  7.50s/it]


multimodal_cot_qwen3: 11it [01:31,  7.57s/it]


multimodal_cot_qwen3: 12it [01:38,  7.39s/it]


multimodal_cot_qwen3: 13it [01:45,  7.35s/it]


multimodal_cot_qwen3: 14it [01:53,  7.40s/it]


multimodal_cot_qwen3: 15it [02:03,  8.28s/it]


multimodal_cot_qwen3: 16it [02:11,  8.20s/it]


multimodal_cot_qwen3: 17it [02:19,  8.09s/it]


multimodal_cot_qwen3: 18it [02:30,  8.87s/it]


multimodal_cot_qwen3: 19it [02:44, 10.51s/it]


multimodal_cot_qwen3: 20it [02:55, 10.64s/it]


mul

  Checkpoint saved at 50 samples





multimodal_cot_qwen3: 51it [08:09,  8.42s/it]


multimodal_cot_qwen3: 52it [08:19,  8.90s/it]


multimodal_cot_qwen3: 53it [08:30,  9.54s/it]


multimodal_cot_qwen3: 54it [08:42, 10.35s/it]


multimodal_cot_qwen3: 55it [08:48,  8.95s/it]


multimodal_cot_qwen3: 56it [08:57,  9.21s/it]


multimodal_cot_qwen3: 57it [09:00,  7.28s/it]


multimodal_cot_qwen3: 58it [09:08,  7.52s/it]


multimodal_cot_qwen3: 59it [09:15,  7.22s/it]


multimodal_cot_qwen3: 60it [09:20,  6.78s/it]


multimodal_cot_qwen3: 61it [09:27,  6.67s/it]


multimodal_cot_qwen3: 62it [09:38,  7.86s/it]


multimodal_cot_qwen3: 63it [09:47,  8.35s/it]


multimodal_cot_qwen3: 64it [09:54,  7.79s/it]


multimodal_cot_qwen3: 65it [10:00,  7.41s/it]


multimodal_cot_qwen3: 66it [10:06,  6.91s/it]


multimodal_cot_qwen3: 67it [10:24, 10.24s/it]


multimodal_cot_qwen3: 68it [10:35, 10.61s/it]


multimodal_cot_qwen3: 69it [10:43,  9.60s/it]


multimodal_cot_qwen3: 70it [10:49,  8.60s/it]


multimodal_cot_qwen3: 71it [11:16, 14

  Checkpoint saved at 100 samples


...

  Checkpoint saved at 950 samples





multimodal_cot_qwen3: 951it [2:19:33,  8.18s/it]


multimodal_cot_qwen3: 952it [2:19:40,  8.07s/it]


multimodal_cot_qwen3: 953it [2:19:49,  8.13s/it]


multimodal_cot_qwen3: 954it [2:19:58,  8.35s/it]


multimodal_cot_qwen3: 955it [2:20:06,  8.31s/it]


multimodal_cot_qwen3: 956it [2:20:13,  7.86s/it]


multimodal_cot_qwen3: 957it [2:20:20,  7.80s/it]


multimodal_cot_qwen3: 958it [2:20:28,  7.76s/it]


multimodal_cot_qwen3: 959it [2:20:35,  7.70s/it]


multimodal_cot_qwen3: 960it [2:20:43,  7.61s/it]


multimodal_cot_qwen3: 961it [2:20:49,  7.31s/it]


multimodal_cot_qwen3: 962it [2:20:58,  7.63s/it]


multimodal_cot_qwen3: 963it [2:21:04,  7.28s/it]


multimodal_cot_qwen3: 964it [2:21:11,  7.05s/it]


multimodal_cot_qwen3: 965it [2:21:19,  7.53s/it]


multimodal_cot_qwen3: 966it [2:21:31,  8.77s/it]


multimodal_cot_qwen3: 967it [2:21:37,  7.96s/it]


multimodal_cot_qwen3: 968it [2:21:44,  7.56s/it]


multimodal_cot_qwen3: 969it [2:21:49,  6.76s/it]


multimodal_cot_qwen3: 970it 

  Checkpoint saved at 1000 samples





multimodal_cot_qwen3: 1001it [2:25:58,  6.02s/it]


multimodal_cot_qwen3: 1002it [2:26:04,  6.05s/it]


multimodal_cot_qwen3: 1003it [2:26:11,  6.11s/it]


multimodal_cot_qwen3: 1004it [2:26:16,  5.83s/it]


multimodal_cot_qwen3: 1005it [2:26:21,  5.74s/it]


multimodal_cot_qwen3: 1006it [2:26:28,  5.98s/it]


multimodal_cot_qwen3: 1007it [2:26:33,  5.81s/it]


multimodal_cot_qwen3: 1008it [2:26:38,  5.62s/it]


multimodal_cot_qwen3: 1009it [2:26:48,  6.74s/it]


multimodal_cot_qwen3: 1010it [2:26:53,  6.32s/it]


multimodal_cot_qwen3: 1011it [2:27:00,  6.60s/it]


multimodal_cot_qwen3: 1012it [2:27:09,  7.09s/it]


multimodal_cot_qwen3: 1013it [2:27:41, 14.72s/it]


multimodal_cot_qwen3: 1014it [2:28:06, 17.80s/it]


multimodal_cot_qwen3: 1015it [2:28:30, 19.52s/it]


multimodal_cot_qwen3: 1016it [2:28:36, 15.42s/it]


multimodal_cot_qwen3: 1017it [2:28:43, 12.90s/it]


multimodal_cot_qwen3: 1018it [2:29:07, 16.35s/it]


multimodal_cot_qwen3: 1019it [2:29:33,  8.81s/it]


Baseline          : multimodal_cot_qwen3
Num samples       : 1019
Parse failure     : 0.007  (7/1019)
Element Acc       : 0.058
Action Acc        : 0.474
Exact Match       : 0.049
Top-3 Element Acc : 0.951
MRR               : 0.951
Task Success Rate : 0.000

Per-action breakdown:
  CLICK    P=0.832  R=0.468  F1=0.599  support=827
  SCROLL   P=0.000  R=0.000  F1=0.000  support=0
  SELECT   P=0.545  R=0.083  F1=0.145  support=72
  SUBMIT   P=0.000  R=0.000  F1=0.000  support=0
  TYPE     P=0.170  R=0.750  F1=0.277  support=120

Metrics saved to: /home/ubuntu/results/multimodal_cot_qwen3_metrics.json


## Cell 10 — Run Baseline: AXTree Only

In [33]:
summary_axtree = run_baseline(
    baseline_name  = "axtree_only_qwen3",
    prompt_fn      = make_prompt_axtree,
    max_new_tokens = CONFIG["max_new_tokens"],
)

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

axtree_only_qwen3: 49it [05:17,  4.78s/it]

                                                                                                                        

axtree_only_qwen3: 50it [05:21,  4.42s/it]                                            | 6/100 [4:33:39<07:04,  4.52s/it]

  Checkpoint saved at 50 samples


axtree_only_qwen3: 99it [12:26,  6.52s/it]

                                                                                                                        

axtree_only_qwen3: 100it [12:31,  5.95s/it]                                           | 6/100 [4:40:49<07:04,  4.52s/it]

  Checkpoint saved at 100 samples


...

  Checkpoint saved at 950 samples


axtree_only_qwen3: 999it [1:34:09,  3.97s/it]

                                                                                                                        

axtree_only_qwen3: 1000it [1:34:12,  3.71s/it]                                        | 6/100 [6:02:30<07:04,  4.52s/it]

  Checkpoint saved at 1000 samples


axtree_only_qwen3: 1019it [1:36:53,  5.70s/it]


Baseline          : axtree_only_qwen3
Num samples       : 1019
Parse failure     : 0.003  (3/1019)
Element Acc       : 0.064
Action Acc        : 0.530
Exact Match       : 0.055
Top-3 Element Acc : 0.955
MRR               : 0.955
Task Success Rate : 0.000

Per-action breakdown:
  CLICK    P=0.897  R=0.495  F1=0.638  support=827
  SELECT   P=0.554  R=0.569  F1=0.562  support=72
  TYPE     P=0.185  R=0.750  F1=0.297  support=120

Metrics saved to: /home/ubuntu/results/axtree_only_qwen3_metrics.json


In [23]:
import json
from pathlib import Path

with open("/home/ubuntu/results/multimodal_cot_qwen3_raw.jsonl") as f:
    rows = [json.loads(l) for l in f]

# Look at a few raw outputs
for row in rows[:5]:
    print(f"RAW OUTPUT:\n{row['raw']}")
    print("-"*60)

RAW OUTPUT:
The task involves finding romantic reggae music from BCD Studio suitable for TikTok in Andorra, but the current page appears to be a TikTok for business dashboard with no relevant content. The best next step is to search or navigate to a music library or BCD Studio’s catalog.
{"element_id": "828", "action": "type", "value": "romantic reggae BCD Studio tiktok Andorra"}
------------------------------------------------------------
RAW OUTPUT:
The task involves finding romantic reggae music from BCD Studio for TikTok use in Andorra, but the current page is unrelated, showing TikTok for business tools. I should navigate to a music or content search platform.
{"element_id": "7010", "action": "click", "value": ""}
------------------------------------------------------------
RAW OUTPUT:
The task is to find romantic reggae music from BCD Studio suitable for TikTok in Andorra, but the current page appears to be a TikTok for business dashboard with no relevant content. The next logica

## Cell 11 — Compare Results

In [34]:
import matplotlib.pyplot as plt

summaries = {
    "Multimodal + CoT": summary_cot,
    "AXTree Only":      summary_axtree,
}

# Print table
print(f"{'Baseline':<25} {'Element Acc':>12} {'Action Acc':>12} {'Exact Match':>12} {'Parse Fail':>12} {'N':>6}")
print("-" * 85)
for name, s in summaries.items():
    print(f"{name:<25} {s['element_acc']:>12.3f} {s['action_acc']:>12.3f} {s['exact_match']:>12.3f} {s['parse_fail']:>12.3f} {s['n']:>6}")

# Bar chart
metrics  = ["element_acc", "action_acc", "exact_match"]
labels   = ["Element Acc", "Action Acc", "Exact Match"]
x        = np.arange(len(metrics))
width    = 0.3

fig, ax = plt.subplots(figsize=(9, 5))
for i, (name, s) in enumerate(summaries.items()):
    vals = [s[m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=name, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.2f}",
                ha="center", va="bottom", fontsize=9)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("Mind2Web Baselines — Qwen3-VL-8B")
ax.legend()
plt.tight_layout()
plt.savefig(Path(CONFIG["results_dir"]) / "baseline_comparison.png", dpi=150)
plt.show()
print("Plot saved.")

Baseline                   Element Acc   Action Acc  Exact Match   Parse Fail      N
-------------------------------------------------------------------------------------


KeyError: 'element_acc'

## Cell 12 — Inspect Individual Predictions

In [35]:
def inspect(baseline_name, n=5, failures_only=False):
    path = Path(CONFIG["results_dir"]) / f"{baseline_name}_raw.jsonl"
    with open(path) as f:
        rows = [json.loads(l) for l in f]

    if failures_only:
        rows = [r for r in rows if r["element_acc"] == 0]
        print(f"Showing {min(n, len(rows))} failures out of {len(rows)} total failures\n")
    else:
        print(f"Showing first {n} samples\n")

    for row in rows[:n]:
        print(f"Task    : {row['instruction'][:80]}")
        print(f"GT      : element={row['gt_element']}  action={row['gt_action']}")
        print(f"Pred    : {row['parsed']}")
        print(f"Correct : element={bool(row['element_acc'])}  action={bool(row['action_acc'])}")
        if "raw" in row:
            print(f"Raw out : {row['raw'][:200]}")
        print("-" * 60)

# Examples:
# inspect("multimodal_cot_qwen3", n=5)
# inspect("axtree_only_qwen3", n=5, failures_only=True)
inspect("multimodal_cot_qwen3", n=3)

Showing first 3 samples

Task    : What are the romantic reggae musics from BCD Studio that can be used in tik tok 
GT      : element=110  action=CLICK
Pred    : {'element_id': '828', 'action': 'type', 'value': 'romantic reggae BCD Studio tiktok Andorra'}
Correct : element=False  action=False
Raw out : The task involves finding romantic reggae music from BCD Studio suitable for TikTok in Andorra, but the current page appears to be a TikTok for business dashboard with no relevant content. The best ne
------------------------------------------------------------
Task    : What are the romantic reggae musics from BCD Studio that can be used in tik tok 
GT      : element=7007  action=CLICK
Pred    : {'element_id': '7010', 'action': 'click', 'value': ''}
Correct : element=False  action=True
Raw out : The task involves finding romantic reggae music from BCD Studio for TikTok use in Andorra, but the current page is unrelated, showing TikTok for business tools. I should navigate to a music or c